# N10 · RL Reward、KL 与 Rollout：为什么 reward 上升也可能是坏事？


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

LLM RL 后训练不是“把 reward 最大化”这么简单。一个健康的 RL run 至少要同时看：reward、KL、entropy、response length、rollout_time、update_time、样本吞吐、reward parser 正确性。

本节目标：理解 PPO/RLHF 常见数据流，知道 KL penalty 的作用，并能排查 reward parser 和 rollout bottleneck。


## 学习地图与版本说明（截至 2026-04-30）

本节把 LLM RL 训练拆成可检查的数据流：prompt 采样、actor rollout、reward 计算、reference logprob/KL、advantage、PPO/GRPO 更新、指标记录。RL 训练最危险的地方是“指标看起来在变好，但模型真的在学坏目标”：reward parser 错、输出变长、KL 失控、样本重复、rollout 太慢，都会让 reward 曲线产生误导。

版本上，本教程参考 OpenAI Spinning Up 的 PPO 解释、Hugging Face TRL 文档、verl quickstart 和 SLiME 文档。RL 框架迭代很快，PPO、GRPO、REINFORCE++、GSPO 等算法和配置名会变；课程重点放在不随版本改变的判断：PPO clipping 约束新旧策略更新幅度，KL penalty 约束 actor 不要偏离 reference 太远，reward 必须先通过 parser 单元测试。

学完本节，你应该能做三类排查：reward 全 0/全 1 时先查 parser 和数据格式；reward 上升但 KL/长度暴涨时怀疑 reward hacking；GPU 利用率低且 rollout_time 高时检查 serving engine、并发、batching 和 trainer/rollout 数据交换，而不是只调学习率。


## 1. LLM RL 数据流

一个简化 RLHF / rule-based RL 流程：

```text
prompt dataset
  → actor policy 生成 response
  → reward function / reward model 打分
  → reference model 计算 KL 约束
  → advantage / returns
  → PPO/GRPO 更新 actor
  → 新 actor 再 rollout
```

系统角色：

- **actor**：正在训练的策略模型。
- **reference**：冻结模型，用于约束策略不要漂移太远。
- **reward**：环境/规则/奖励模型给的分数。
- **rollout engine**：负责生成样本，常是吞吐瓶颈。
- **trainer**：负责反向传播更新 actor。


## 2. Reward parser 是第一道生死线

在 GSM8K 这类数学任务里，常用规则奖励：抽取最终答案数字，与标准答案比较。如果 parser 抽错，RL 会优化错误目标。

常见错误：

- 抽到中间数字，不是最终答案。
- 没处理逗号、小数、负号、单位。
- 模型输出多个候选答案。
- 标准答案格式变化，例如 `#### 42`。
- reward 全 0 或全 1，advantage 没有效信号。


In [ ]:
import re
import pandas as pd

def extract_final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else None

def reward(prediction, target):
    return float(extract_final_number(prediction) == extract_final_number(target))

cases = [
    {"prediction": "先算 40+2=42，所以答案是 42。", "target": "#### 42"},
    {"prediction": "我看到题里有 40 和 2。答案：41", "target": "#### 42"},
    {"prediction": "1,000 + 20 = 1,020", "target": "#### 1020"},
    {"prediction": "没有给出答案", "target": "#### 3"},
]
for c in cases:
    c["pred_final"] = extract_final_number(c["prediction"])
    c["target_final"] = extract_final_number(c["target"])
    c["reward"] = reward(c["prediction"], c["target"])

display(pd.DataFrame(cases))


## 3. KL penalty 的本质：别为了奖励跑飞

RL 中，actor 可能找到“骗 reward”的方式：输出很长、格式奇怪、过拟合 reward model、偏离原模型语言能力。KL penalty 用 reference model 约束 actor：不要离参考分布太远。

常见目标可粗略理解为：

```text
objective ≈ reward_score - beta × KL(actor || reference)
```

注意：PPO clipping 和 RLHF objective 里的 KL penalty 不是同一个东西。PPO clipping 限制新旧策略更新幅度；KL penalty 通常约束当前策略与 reference model 的距离。


In [ ]:
import numpy as np

def kl_divergence(p, q):
    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)
    return float(np.sum(p * (np.log(p + 1e-12) - np.log(q + 1e-12))))

reference = [0.6, 0.3, 0.1]
policies = {
    "接近 reference": [0.55, 0.35, 0.10],
    "轻微漂移": [0.75, 0.20, 0.05],
    "严重漂移": [0.98, 0.01, 0.01],
}
rows = []
for name, p in policies.items():
    kl = kl_divergence(p, reference)
    rows.append({"policy": name, "KL": kl, "reward": 1.0, "objective_beta0.1": 1.0 - 0.1 * kl, "objective_beta1.0": 1.0 - 1.0 * kl})

display(pd.DataFrame(rows).round(4))


## 4. response length 为什么必须看？

Reward 上升但 response length 同时暴涨，可能意味着：

- 模型学会输出冗长推理来增加命中概率。
- reward parser 被最后一个数字欺骗。
- KL 太低，策略漂移。
- stop token / EOS 配置错误。
- rollout max length 太大，吞吐下降。

所以 L10/L11 报告必须同时看 `reward_mean`、`kl_mean`、`response_len_mean`、`rollout_time_sec`。


In [ ]:
rows = [
    {"run": "A", "reward_mean": 0.42, "kl_mean": 0.03, "response_len": 64, "判断": "健康但仍需看趋势"},
    {"run": "B", "reward_mean": 0.60, "kl_mean": 1.20, "response_len": 240, "判断": "危险：reward 上升伴随 KL/长度爆炸"},
    {"run": "C", "reward_mean": 0.05, "kl_mean": 0.01, "response_len": 20, "判断": "可能 reward parser 全错或 rollout 太弱"},
]
display(pd.DataFrame(rows))


## 5. Rollout bottleneck：RL 系统慢在哪里？

RL 后训练常常不是 backward 最慢，而是 rollout 最慢：生成 response 需要 serving 引擎，且每轮训练要生成大量样本。

系统瓶颈包括：

- rollout GPU 太少。
- prompt/output 太长。
- SGLang/vLLM 服务吞吐不足。
- actor 更新后 weight sync 太慢。
- reward function 太慢或串行。
- 数据 buffer/queue 设计差。

这就是为什么 L11 SLiME 要比较 actor/rollout GPU split 和 weight sync。


In [ ]:
def rl_step_time(rollout_samples=1024, generation_sps=128, update_sec=30, reward_sec=5, sync_sec=8):
    rollout_sec = rollout_samples / generation_sps
    total = rollout_sec + update_sec + reward_sec + sync_sec
    return {"rollout_sec": rollout_sec, "update_sec": update_sec, "reward_sec": reward_sec, "sync_sec": sync_sec, "total_sec": total, "rollout_pct": rollout_sec / total}

pd.DataFrame([
    rl_step_time(generation_sps=64),
    rl_step_time(generation_sps=256),
    rl_step_time(generation_sps=512, sync_sec=25),
]).round(2)


## 6. 与本课程的连接

- L10 verl：先做 GSM8K prompt + reward self-test + RL metrics。
- L11 SLiME：Megatron actor training + SGLang rollout + weight sync。
- L12 Capstone：回答“reward 不涨、KL 爆炸、TTFT 回归、resume 失败”这类综合问题。
- Debug ticket：`verl_reward_parse_001`、`verl_kl_explosion_002`、`slime_rollout_bottleneck_001`、`slime_reward_collapse_002`。


## 7. 企业面试/工程判断痛点题（带答案）

### 题 1：reward_mean 上升一定代表 RL 训练变好吗？

**答案解析：** 不一定。要同时看 KL、response length、reward parser、人工样本、成功率。如果 reward 上升伴随 KL/长度爆炸，可能是 reward hacking。

### 题 2：PPO clipping 和 KL penalty 是一回事吗？

**答案解析：** 不是。PPO clipping 限制新旧策略更新带来的目标收益；RLHF KL penalty 通常约束 actor 与 reference model 的距离。

### 题 3：GSM8K rule reward 全 0，第一步查什么？

**答案解析：** 查 reward parser self-test：是否正确抽取 `####` 后答案，是否抽到最终数字，是否处理逗号/负号/小数。不要先调学习率。

### 题 4：rollout_time 远大于 update_time，应该加训练 GPU 还是 rollout GPU？

**答案解析：** 优先增加/优化 rollout 资源或 serving 吞吐，而不是盲目加 actor 训练 GPU。否则 actor 更快也会等 rollout。

### 题 5：KL 很低但 reward 不涨，可能是什么原因？

**答案解析：** 策略几乎没动，可能学习率太低、advantage 信号弱、reward 全零、采样温度/探索不足、更新被 clipping/配置限制。


## 参考资料

- OpenAI Spinning Up PPO: https://spinningup.openai.com/en/latest/algorithms/ppo.html
- Hugging Face TRL PPO Trainer metrics: https://huggingface.co/docs/trl/en/ppo_trainer
- verl GSM8K PPO quickstart: https://verl.readthedocs.io/en/v0.5.x/start/quickstart.html
- SLiME documentation: https://thudm.github.io/slime/
